# TMJ Binary Position Classifier — Detector-Based Crops

Ноутбук для обучения **бинарного классификатора** положения ВНЧС (центральное / нецентральное).

**Approach A**: NIfTI-кропы 128³ вокруг предсказаний детектора (вместо центральных кропов).  
**Approach B**: `BinaryFocalLoss` + калибровка порога через Youden's J.

Поддерживаемые среды: **Yandex DataSphere** (основная) · **Google Colab** · **Local**

In [ ]:
%pip install -q --upgrade scipy tqdm nibabel scikit-learn

import subprocess, sys
from pathlib import Path

# Download detector model from GitHub Release (one-time)
DETECTOR_URL = "https://github.com/tzopiz/MasterProject/releases/download/detector-v1/best_model.pth"
DETECTOR_PATH_DS = Path("/detector/models/best_model.pth")

if Path("/home/jupyter").exists():  # DataSphere
    DETECTOR_PATH_DS.parent.mkdir(parents=True, exist_ok=True)
    if not DETECTOR_PATH_DS.exists():
        print("Скачиваю детектор (165 MB)...")
        r = subprocess.run(["wget", "-q", "--show-progress", "-O", str(DETECTOR_PATH_DS), DETECTOR_URL])
        print(f"Готово: {DETECTOR_PATH_DS.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"Детектор: {DETECTOR_PATH_DS} ({DETECTOR_PATH_DS.stat().st_size / 1e6:.1f} MB)")

In [ ]:
import os, sys
from pathlib import Path

IN_DATASPHERE = Path("/home/jupyter").exists()
IN_COLAB = "google.colab" in sys.modules
env_name = "DataSphere" if IN_DATASPHERE else ("Colab" if IN_COLAB else "Local")
print(f"Среда: {env_name}")

if IN_DATASPHERE:
    DATA_ROOT     = Path("/home/jupyter/datasets/tmj_data")
    WORK_ROOT     = Path("/detector")
    DETECTOR_PATH = Path("/detector/models/best_model.pth")
elif IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DATA_ROOT     = Path("/content/drive/MyDrive/tmj_data")
    WORK_ROOT     = Path("/content")
    DETECTOR_PATH = DATA_ROOT / "models" / "best_model.pth"
else:
    DATA_ROOT     = Path("../data")
    WORK_ROOT     = Path("..")
    DETECTOR_PATH = Path("../experiments/detector_20251126_003305/best_model.pth")

DATASET_ROOT  = DATA_ROOT / "dataset_public"
MANIFEST_PATH = DATASET_ROOT / "manifest_private.json"
LABELS_PATH   = DATA_ROOT / "tmj_position_labels.json"
CROPS_DIR     = WORK_ROOT / "detector_crops"
OUTPUT_DIR    = WORK_ROOT / "experiments"

CROPS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT     : {DATA_ROOT}  exists={DATA_ROOT.exists()}")
print(f"MANIFEST_PATH : {MANIFEST_PATH}  exists={MANIFEST_PATH.exists()}")
print(f"LABELS_PATH   : {LABELS_PATH}  exists={LABELS_PATH.exists()}")
print(f"DETECTOR_PATH : {DETECTOR_PATH}  exists={DETECTOR_PATH.exists()}")
print(f"CROPS_DIR     : {CROPS_DIR}")

In [ ]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda"); print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps"); print("MPS")
else:
    device = torch.device("cpu"); print("CPU")
print(f"PyTorch: {torch.__version__}")

## 2. Label Table

In [ ]:
import json, random, logging
from pathlib import Path
from typing import Dict, List, Tuple

logger = logging.getLogger("tmj")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")


def map_sagittal(code):
    if code not in (1, 2, 3): raise ValueError(f"Invalid sagittal code: {code}")
    return code - 1


def map_frontal(code):
    if code not in (4, 5, 6): raise ValueError(f"Invalid frontal code: {code}")
    return code - 4


def build_index(manifest_path, labels_path, dataset_root, cache_path=None):
    with open(manifest_path, "r", encoding="utf-8") as f: manifest = json.load(f)
    with open(labels_path, "r", encoding="utf-8") as f: labels_data = json.load(f)
    label_by_name = {p["name_raw"].strip(): p["labels"] for p in labels_data["patients"]}
    records, skipped = [], 0
    for study in manifest["studies"]:
        name = study["patient_name"].strip()
        if name not in label_by_name: skipped += 1; continue
        lbl = label_by_name[name]
        records.append({
            "study_id": study["study_id"],
            "dicom_dir": str(Path(dataset_root) / study["study_id"]),
            "patient_name": name,
            "sag_right": map_sagittal(lbl["sagittal"]["right"]),
            "sag_left":  map_sagittal(lbl["sagittal"]["left"]),
            "fr_right":  map_frontal(lbl["frontal"]["right"]),
            "fr_left":   map_frontal(lbl["frontal"]["left"]),
        })
    logger.info("build_index: %d matched, %d skipped", len(records), skipped)
    return records


def binarize_labels(records, crop_dir):
    crop_dir = Path(crop_dir).resolve()
    out = []
    for rec in records:
        for side in ("left", "right"):
            out.append({
                "study_id": rec["study_id"],
                "patient_name": rec["patient_name"],
                "side": side,
                "sag": 0 if rec[f"sag_{side}"] == 0 else 1,
                "fr":  0 if rec[f"fr_{side}"]  == 0 else 1,
                "crop_path": str(crop_dir / rec["study_id"] / f"{rec['study_id']}_{side}.nii.gz"),
            })
    logger.info("binarize_labels: %d → %d binary records", len(records), len(out))
    return out


def split_by_patient(records, split_ratio=0.8, seed=42):
    patients = sorted(set(r["patient_name"] for r in records))
    rng = random.Random(seed); rng.shuffle(patients)
    n = len(patients)
    idx = min(max(1, int(n * split_ratio)), n - 1)
    train_p = set(patients[:idx]); val_p = set(patients[idx:])
    train = [r for r in records if r["patient_name"] in train_p]
    val   = [r for r in records if r["patient_name"] in val_p]
    logger.info("split: train=%d val=%d", len(train), len(val))
    return train, val


all_records = build_index(str(MANIFEST_PATH), str(LABELS_PATH), str(DATASET_ROOT))
binary_records = binarize_labels(all_records, crop_dir=str(CROPS_DIR))

from collections import Counter
sag_dist = Counter(r["sag"] for r in binary_records)
fr_dist  = Counter(r["fr"]  for r in binary_records)
print(f"Записей: {len(all_records)} исследований → {len(binary_records)} кропов")
print(f"Sagittal: central={sag_dist[0]} non-central={sag_dist[1]}")
print(f"Frontal:  central={fr_dist[0]}  non-central={fr_dist[1]}")

In [ ]:
SPLIT_RATIO = 0.8
train_records, val_records = split_by_patient(binary_records, split_ratio=SPLIT_RATIO, seed=42)
print(f"Train: {len(train_records)}  Val: {len(val_records)}")

## 3. Preprocessing: Detector → Crops (один раз)

In [ ]:
import torch.nn as nn
import numpy as np
import pydicom
from scipy import ndimage
import nibabel as nib
from tqdm.notebook import tqdm


# ── Detector model (inline) ────────────────────────────────────────────────
class TMJDetectorLarge(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        def _cb(ic, oc):
            return nn.Sequential(
                nn.Conv3d(ic, oc, 3, padding=1), nn.BatchNorm3d(oc), nn.ReLU(inplace=True),
                nn.Conv3d(oc, oc, 3, padding=1), nn.BatchNorm3d(oc), nn.ReLU(inplace=True),
            )
        self.conv1 = _cb(in_channels, 32); self.pool1 = nn.MaxPool3d(2)
        self.conv2 = _cb(32, 64);  self.pool2 = nn.MaxPool3d(2)
        self.conv3 = _cb(64, 128); self.pool3 = nn.MaxPool3d(2)
        self.conv4 = _cb(128, 256); self.pool4 = nn.MaxPool3d(2)
        self.conv5 = _cb(256, 512)
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.fc_left  = nn.Sequential(nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(256, 3), nn.Sigmoid())
        self.fc_right = nn.Sequential(nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(256, 3), nn.Sigmoid())

    def forward(self, x):
        x = self.pool1(self.conv1(x)); x = self.pool2(self.conv2(x))
        x = self.pool3(self.conv3(x)); x = self.pool4(self.conv4(x))
        x = self.conv5(x); x = self.global_pool(x); x = x.view(x.size(0), -1)
        return torch.cat([self.fc_left(x), self.fc_right(x)], dim=1)


# ── Crop generation helpers (inline) ──────────────────────────────────────
def load_dicom_volume(dicom_dir):
    files = sorted(Path(dicom_dir).glob("*.dcm"))
    if not files: raise FileNotFoundError(f"No .dcm in {dicom_dir}")
    slices = [pydicom.dcmread(str(f)) for f in files]
    try: slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))
    except: slices.sort(key=lambda s: int(s.InstanceNumber))
    planes = []
    for s in slices:
        arr = s.pixel_array.astype(np.float32)
        arr = arr * float(getattr(s, "RescaleSlope", 1.0)) + float(getattr(s, "RescaleIntercept", 0.0))
        planes.append(arr)
    return np.stack(planes, axis=0)


def preprocess_volume(volume):
    p2, p98 = np.percentile(volume, [2, 98])
    volume = np.clip(volume, p2, p98)
    if volume.max() > volume.min():
        volume = (volume - volume.min()) / (volume.max() - volume.min())
    return volume.astype(np.float32)


def predict_coords(model, volume, downsample=6, device="cpu"):
    v = preprocess_volume(volume)
    D, H, W = v.shape
    nD, nH, nW = D // downsample, H // downsample, W // downsample
    t = torch.nn.functional.interpolate(
        torch.from_numpy(v[None, None]).float(), size=(nD, nH, nW),
        mode="trilinear", align_corners=False
    )[0, 0].numpy()
    inp = torch.from_numpy(t).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad(): pred = model(inp).cpu().numpy()[0]
    coords = np.array([pred[0]*nD, pred[1]*nH, pred[2]*nW, pred[3]*nD, pred[4]*nH, pred[5]*nW])
    coords[[0, 3]] *= downsample; coords[[1, 4]] *= downsample; coords[[2, 5]] *= downsample
    return {"left": coords[:3].astype(int), "right": coords[3:].astype(int)}


def extract_crop(volume, center, size=128):
    D, H, W = volume.shape; half = size // 2
    z, y, x = center
    zs, ze = max(0, z - half), min(D, z + half)
    ys, ye = max(0, y - half), min(H, y + half)
    xs, xe = max(0, x - half), min(W, x + half)
    crop = volume[zs:ze, ys:ye, xs:xe]
    if crop.shape != (size, size, size):
        pad = np.zeros((size, size, size), dtype=crop.dtype)
        pz = (size - crop.shape[0]) // 2
        py = (size - crop.shape[1]) // 2
        px = (size - crop.shape[2]) // 2
        pad[pz:pz+crop.shape[0], py:py+crop.shape[1], px:px+crop.shape[2]] = crop
        crop = pad
    return crop


# ── Main crop generation loop ──────────────────────────────────────────────
existing = list(CROPS_DIR.rglob("*.nii.gz"))
print(f"Кропов уже есть: {len(existing)}, ожидается: {len(all_records)*2}")

if len(existing) < len(all_records) * 2:
    assert DETECTOR_PATH.exists(), f"Детектор не найден: {DETECTOR_PATH}"

    # Load detector
    ckpt = torch.load(DETECTOR_PATH, map_location="cpu", weights_only=False)
    det_model = TMJDetectorLarge().to(device)
    det_model.load_state_dict(ckpt["model_state_dict"])
    det_model.eval()
    print(f"Детектор загружен (эпоха {ckpt['epoch']}, MAE={ckpt.get('best_val_mae', '?')})")

    for rec in tqdm(all_records, desc="Generating crops"):
        study_id = rec["study_id"]
        out_dir   = CROPS_DIR / study_id
        left_path  = out_dir / f"{study_id}_left.nii.gz"
        right_path = out_dir / f"{study_id}_right.nii.gz"
        if left_path.exists() and right_path.exists(): continue

        out_dir.mkdir(parents=True, exist_ok=True)
        volume = load_dicom_volume(rec["dicom_dir"])
        coords = predict_coords(det_model, volume, device=str(device).split(":")[0])

        for side in ("left", "right"):
            crop = extract_crop(volume, coords[side])
            path = out_dir / f"{study_id}_{side}.nii.gz"
            nib.save(nib.Nifti1Image(crop, affine=np.eye(4)), str(path))

    print("Готово!")
else:
    print("Кропы уже есть — пропускаем.")

In [ ]:
missing = [r for r in binary_records if not Path(r["crop_path"]).exists()]
print(f"Кропов не найдено: {len(missing)} из {len(binary_records)}")
if missing:
    print("Примеры:", [r["crop_path"] for r in missing[:3]])

## 4. Dataset & DataLoader

In [ ]:
import random as _random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import nibabel as nib


class TMJBinaryPositionDataset(Dataset):
    def __init__(self, records, is_train=False):
        self.records = records
        self.is_train = is_train

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = nib.load(rec["crop_path"])
        vol = np.asarray(img.dataobj, dtype=np.float32)
        # normalize [2nd, 98th] percentile → [0, 1]
        p2, p98 = np.percentile(vol, [2, 98])
        vol = np.clip(vol, p2, p98)
        denom = p98 - p2
        vol = (vol - p2) / denom if denom > 0 else np.zeros_like(vol)
        if self.is_train:
            for ax in range(3):
                if _random.random() < 0.5:
                    vol = np.flip(vol, axis=ax).copy()
        tensor = torch.from_numpy(vol).float().unsqueeze(0)  # (1, D, H, W)
        labels = torch.tensor([rec["sag"], rec["fr"]], dtype=torch.long)
        return tensor, labels


BATCH_SIZE   = 4
NUM_WORKERS  = 0

train_ds = TMJBinaryPositionDataset(train_records, is_train=True)
val_ds   = TMJBinaryPositionDataset(val_records,   is_train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_ds)} сэмплов, {len(train_loader)} батчей")
print(f"Val:   {len(val_ds)} сэмплов, {len(val_loader)} батчей")

# smoke test
vol, lbl = next(iter(train_loader))
print(f"volume: {vol.shape} [{vol.min():.2f}, {vol.max():.2f}]  labels: {lbl.shape}")

## 5. Model

In [ ]:
import torch.nn as nn
from typing import List, Optional, Tuple


def _conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm3d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool3d(2, 2),
    )


class TMJBinaryPositionClassifier(nn.Module):
    def __init__(self, in_channels=1, features=None, fc_hidden=256, dropout=0.5):
        super().__init__()
        if features is None: features = [16, 32, 64, 128]
        blocks, prev = [], in_channels
        for oc in features:
            blocks.append(_conv_block(prev, oc)); prev = oc
        self.backbone    = nn.Sequential(*blocks)
        self.global_pool = nn.AdaptiveAvgPool3d(1)

        def _head():
            return nn.Sequential(
                nn.Linear(prev, fc_hidden),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(fc_hidden, 1),
            )
        self.head_sag = _head()
        self.head_fr  = _head()

    def forward(self, x):
        f = self.backbone(x)
        f = self.global_pool(f).view(f.size(0), -1)
        return self.head_sag(f), self.head_fr(f)


model = TMJBinaryPositionClassifier().to(device)
n = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n/1e6:.2f}M")

with torch.no_grad():
    s, f = model(torch.randn(1, 1, 32, 48, 48).to(device))
print(f"sag: {s.shape}, fr: {f.shape}")

## 6. Training

In [ ]:
import torch.nn.functional as F


class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        if reduction not in ("mean", "sum", "none"):
            raise ValueError(f"Bad reduction: {reduction}")
        self.gamma, self.alpha, self.reduction = gamma, alpha, reduction

    def forward(self, logits, targets):
        if logits.dim() == 2 and logits.shape[1] == 1: logits = logits.squeeze(1)
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p   = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = (1 - p_t).pow(self.gamma) * bce
        if self.alpha is not None:
            loss = (self.alpha * targets + (1 - self.alpha) * (1 - targets)) * loss
        return loss.mean() if self.reduction == "mean" else (
            loss.sum() if self.reduction == "sum" else loss
        )


# ── Hyperparameters ────────────────────────────────────────────────────────
EPOCHS          = 100
LR              = 1e-4
WEIGHT_DECAY    = 1e-5
LR_PATIENCE     = 10
EARLY_STOPPING  = 30
GAMMA           = 2.0
HEAD_NAMES      = ["sag", "fr"]


# ── Auto-compute class weights (alpha) ────────────────────────────────────
def compute_class_weights(loader):
    counts = {n: {0: 0, 1: 0} for n in HEAD_NAMES}
    for _, labels in loader:
        for i, n in enumerate(HEAD_NAMES):
            for c in (0, 1):
                counts[n][c] += (labels[:, i] == c).sum().item()
    alphas = {}
    for n in HEAD_NAMES:
        total = counts[n][0] + counts[n][1]
        alphas[n] = counts[n][0] / total if total > 0 else 0.5
        print(f"[{n}] 0={counts[n][0]} 1={counts[n][1]}  α={alphas[n]:.3f}")
    return alphas


alphas = compute_class_weights(train_loader)

In [ ]:
import datetime, json as _json2
import torch.optim as optim

criteria  = {n: BinaryFocalLoss(gamma=GAMMA, alpha=alphas[n]) for n in HEAD_NAMES}
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=LR_PATIENCE)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
exp_dir   = OUTPUT_DIR / f"binary_position_{timestamp}"
exp_dir.mkdir(parents=True, exist_ok=True)

config = {
    "epochs": EPOCHS, "lr": LR, "gamma": GAMMA, "alphas": alphas,
    "batch_size": BATCH_SIZE, "split_ratio": SPLIT_RATIO,
    "train_samples": len(train_ds), "val_samples": len(val_ds),
}
with open(exp_dir / "config.json", "w") as f: _json2.dump(config, f, indent=2)
print(f"Эксперимент: {exp_dir}")

In [ ]:
from tqdm.notebook import tqdm


def compute_metrics(sl, fl, labels, thresh=(0.5, 0.5)):
    m = {}
    for i, (n, lo, th) in enumerate(zip(HEAD_NAMES, [sl, fl], thresh)):
        preds = (torch.sigmoid(lo.squeeze(1)) >= th).long()
        m[f"acc_{n}"] = (preds == labels[:, i]).float().mean().item()
    m["mean_accuracy"] = float(np.mean([m[f"acc_{n}"] for n in HEAD_NAMES]))
    return m


def run_epoch(train):
    model.train() if train else model.eval()
    loader = train_loader if train else val_loader
    tag    = "Train" if train else "Val  "
    rl, all_m = 0.0, []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for vols, labels in tqdm(loader, desc=tag, leave=False):
            vols, labels = vols.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            sl, fl = model(vols)
            loss = criteria["sag"](sl, labels[:, 0].float()) + criteria["fr"](fl, labels[:, 1].float())
            if train: loss.backward(); optimizer.step()
            m = compute_metrics(sl, fl, labels); m["loss"] = loss.item()
            all_m.append(m); rl += loss.item()
    avg = {k: float(np.mean([m[k] for m in all_m])) for k in all_m[0]}
    avg["loss"] = rl / len(loader)
    return avg


best_val_acc = -1.0; no_imp = 0; history = []
best_path = exp_dir / "best_model.pth"

for epoch in range(1, EPOCHS + 1):
    tr  = run_epoch(True)
    val = run_epoch(False)
    scheduler.step(val["mean_accuracy"])
    lr_now = optimizer.param_groups[0]["lr"]
    print(
        f"[{epoch:3d}/{EPOCHS}] "
        f"train acc={tr['mean_accuracy']:.3f}(s={tr['acc_sag']:.3f} f={tr['acc_fr']:.3f}) | "
        f"val acc={val['mean_accuracy']:.3f}(s={val['acc_sag']:.3f} f={val['acc_fr']:.3f}) "
        f"loss={val['loss']:.4f} lr={lr_now:.2e}"
    )
    row = {"epoch": epoch, "lr": lr_now}
    row.update({f"train_{k}": v for k, v in tr.items()})
    row.update({f"val_{k}": v for k, v in val.items()})
    history.append(row)
    with open(exp_dir / "metrics.jsonl", "a") as f: f.write(_json2.dumps(row) + "\n")
    if val["mean_accuracy"] > best_val_acc:
        best_val_acc = val["mean_accuracy"]; no_imp = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_val_accuracy": best_val_acc,
            "val_metrics": val,
        }, best_path)
        print(f"  ✓ best model (acc={best_val_acc:.3f})")
    else:
        no_imp += 1
        if EARLY_STOPPING > 0 and no_imp >= EARLY_STOPPING:
            print(f"  Early stop at {epoch}"); break

print(f"\nГотово. Best val acc: {best_val_acc:.3f}")

## 7. Threshold Calibration

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

ckpt = torch.load(best_path, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
print(f"Best epoch: {ckpt['epoch']}, val acc: {ckpt['best_val_accuracy']:.3f}")

all_probs  = {n: [] for n in HEAD_NAMES}
all_labels = {n: [] for n in HEAD_NAMES}

with torch.no_grad():
    for vols, labels in val_loader:
        sl, fl = model(vols.to(device))
        for i, n in enumerate(HEAD_NAMES):
            lo = [sl, fl][i]
            all_probs[n].extend(torch.sigmoid(lo.squeeze(1)).cpu().tolist())
            all_labels[n].extend(labels[:, i].tolist())

calibration = {"optimal_thresholds": {}, "auc_roc": {}, "accuracy_at_threshold": {}}

for n in HEAD_NAMES:
    probs = np.array(all_probs[n]); labs = np.array(all_labels[n])
    if len(np.unique(labs)) < 2:
        print(f"[{n}] один класс → thresh=0.5")
        calibration["optimal_thresholds"][n] = 0.5
        calibration["auc_roc"][n] = None
        calibration["accuracy_at_threshold"][n] = float(np.mean((probs >= 0.5) == labs))
        continue
    fpr, tpr, ths = roc_curve(labs, probs)
    auc = roc_auc_score(labs, probs)
    best_idx = int(np.argmax(tpr - fpr))
    th  = float(ths[best_idx])
    acc = float(np.mean((probs >= th) == labs))
    calibration["optimal_thresholds"][n] = round(th, 4)
    calibration["auc_roc"][n]            = round(auc, 4)
    calibration["accuracy_at_threshold"][n] = round(acc, 4)
    print(f"[{n}] AUC={auc:.3f}  thresh={th:.3f}  acc={acc:.3f}")

with open(exp_dir / "config.json") as f: cfg = _json2.load(f)
cfg.update(calibration); cfg["best_val_accuracy"] = best_val_acc
with open(exp_dir / "config.json", "w") as f: _json2.dump(cfg, f, indent=2)
print(f"\nПороги: {calibration['optimal_thresholds']}")
print(f"AUC-ROC: {calibration['auc_roc']}")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

ep = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(ep, [h["train_loss"] for h in history], label="Train")
axes[0].plot(ep, [h["val_loss"]   for h in history], label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, [h["train_mean_accuracy"] for h in history], label="Train")
axes[1].plot(ep, [h["val_mean_accuracy"]   for h in history], label="Val")
axes[1].axhline(0.68, color="red", linestyle="--", alpha=0.5, label="Baseline v5")
axes[1].set_title("Mean Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, [h["val_acc_sag"] for h in history], label="Sag")
axes[2].plot(ep, [h["val_acc_fr"]  for h in history], label="Fr")
axes[2].set_title("Val per Head"); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); fig.savefig(exp_dir / "training_curves.png", dpi=150); plt.show()

# ── ROC curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, n in zip(axes, HEAD_NAMES):
    p = np.array(all_probs[n]); l = np.array(all_labels[n])
    if len(np.unique(l)) < 2:
        ax.text(0.5, 0.5, "Один класс", ha="center", transform=ax.transAxes); continue
    fpr, tpr, _ = roc_curve(l, p); a = roc_auc_score(l, p)
    ax.plot(fpr, tpr, lw=2, label=f"AUC={a:.3f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_title(f"{n.upper()} thresh={calibration['optimal_thresholds'][n]}")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(exp_dir / "roc_curves.png", dpi=150); plt.show()

# ── Confusion matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, n in zip(axes, HEAD_NAMES):
    p  = np.array(all_probs[n]); l = np.array(all_labels[n])
    th = calibration["optimal_thresholds"][n]
    preds = (p >= th).astype(int)
    cm = confusion_matrix(l, preds, labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["central", "non-central"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["central", "non-central"])
    ax.set_title(f"{n.upper()} (thresh={th})")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    print(f"\n=== {n.upper()} ===")
    print(classification_report(l, preds, labels=[0, 1], target_names=["central", "non-central"]))
plt.tight_layout(); fig.savefig(exp_dir / "confusion_matrices.png", dpi=150); plt.show()

## 9. Export

In [ ]:
analysis = {
    "config": cfg,
    "total_epochs": len(history),
    "best_epoch": ckpt["epoch"],
    "best_val_accuracy": best_val_acc,
    "calibration": calibration,
    "history": history,
}
ap = exp_dir / "training_analysis.json"
with open(ap, "w") as f: _json2.dump(analysis, f, indent=2, ensure_ascii=False)
print(f"Analysis: {ap}")
print(f"Model:    {best_path}")
print(f"\nИтог — Best val acc: {best_val_acc:.3f} (baseline: 0.680)")
for n in HEAD_NAMES:
    print(f"  [{n}] AUC={calibration['auc_roc'][n]}  thresh={calibration['optimal_thresholds'][n]}")